# 雪音畫布 Yukine Palette · 遠端運算 Colab

讓 Colab 像 RunningHub 一樣，被雪音畫布直接呼叫：**畫布在本機組裝完整 ComfyUI API 圖**，透過認證 API 送到這裡的 ComfyUI 執行，再把結果下載回畫布。

- **不需要上傳任何工作流 JSON**（舊版 notebook 的「雙 JSON 上傳」已移除）。素材由畫布前端上傳。
- 支援畫布所有節點：MiniMax H3 影片（一採／二採／Motion Context）、**SCAIL-2 動作遷移／人物替換**、圖片（Z-Image / Flux2 Klein / MOSS / Anima）、YuE2 音樂、Qwen3 / IndexTTS / Breeze TTS、自訂工作流。
- 模型與節點**依需要勾選**才安裝；來源優先使用 Comfy-Org 官方 Hugging Face，每個模型放到 ComfyUI 對應的類別資料夾（可改位置）。
- 預設一併安裝 **ComfyUI-Manager**，可在 ComfyUI 網頁介面直接安裝其他節點套件。

**順序：Cell 1 選擇 → Cell 2 安裝 → Cell 3 下載 → Cell 4 啟動 ComfyUI → Cell 5 API + tunnel → 畫布填入 H3_API_URL / token。**

| 類型 | 設定 |
|---|---|
| GPU | 建議 L4 / A100 / H100、高 RAM。H3 與 SCAIL-2 14B 都需要大量 VRAM 與磁碟。 |
| Secrets（選用） | `HF_TOKEN`（需授權的 HF 模型）、`H3_API_TOKEN`（≥24 字元；沒設定可在 Cell 5 自動產生） |
| 舊版 notebook | `MiniMax_H3_MotionContext_Remote_20261102.ipynb` 仍保留手動生成介面與 `/generate` API；畫布遠端運算請用本檔。 |

## Cell 1 — 選擇功能、節點與模型
勾選功能會自動預勾「畫布預設會用到的模型」；選用變體（pruned、fp8、社群 hybrid…）需手動勾。可以只裝需要的部分，之後回來追加再重跑 Cell 3。
- **模型根目錄**可改到 Google Drive（並勾選 MOUNT_DRIVE）以重複使用模型。
- 單一模型右欄可填另一個資料夾（相對根目錄或絕對路徑）；不在 ComfyUI/models 底下的資料夾會自動寫入 `extra_model_paths.yaml`。
- `EXTRA_MODELS` / `CUSTOM_NODE_REPOS` 可加入個人 LoRA 或其他節點（例如畫布預設的 Anima 社群微調、Z-Image LoRA 不在官方來源，請自行加入）。

In [ ]:
#@title Cell 1 — 選擇要安裝的節點套件與模型（不會立即下載）
"""Pick only what you need. Nothing is downloaded until Cell 2 / Cell 3.

Every model lands in the ComfyUI category folder that the canvas graphs expect
(diffusion_models, text_encoders, vae, loras, ...). You may change the root
folder or any single model's target folder; folders outside ComfyUI/models are
registered automatically through extra_model_paths.yaml in Cell 4.
"""
import json
import os
from pathlib import Path

import ipywidgets as W
from IPython.display import display

COMFY = Path('/content/ComfyUI')
SELECTION_PATH = Path('/content/h3_canvas_selection.json')

#@markdown 模型根目錄（預設為 ComfyUI/models；可改成 Google Drive 路徑以保留模型，例如 /content/drive/MyDrive/ComfyUI/models）
MODEL_ROOT = '/content/ComfyUI/models'  #@param {type:'string'}
#@markdown 掛載 Google Drive（模型根目錄在 Drive 時請勾選）
MOUNT_DRIVE = False  #@param {type:'boolean'}
#@markdown 安裝 ComfyUI-Manager（可在 ComfyUI 網頁介面直接安裝其他節點套件）
INSTALL_MANAGER = True  #@param {type:'boolean'}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# Custom node packs. ComfyUI core already provides SCAIL-2, SAM3, YuE2, Z-Image,
# Flux2, Anima, MOSS, ManualSigmas, SeedNode, ResolutionSelector and friends.
# ---------------------------------------------------------------------------
NODE_PACKS = {
    'VHS': ('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git', 'ComfyUI-VideoHelperSuite'),
    'T8': ('https://github.com/T8mars/comfyui-minimax-h3-audio-T8.git', 'comfyui-minimax-h3-audio-T8'),
    'MotionContext': ('https://github.com/NikoDemon80/ComfyUI-H3-Motion-Context.git', 'ComfyUI-H3-Motion-Context'),
    'KJNodes': ('https://github.com/kijai/ComfyUI-KJNodes.git', 'ComfyUI-KJNodes'),
    'ListHelper': ('https://github.com/dseditor/ComfyUI-ListHelper.git', 'ComfyUI-ListHelper'),
    'GoohaiTools': ('https://github.com/goohai/Goohaitools-comfyui.git', 'Goohaitools-comfyui'),
    'LBH_Upscaler': ('https://github.com/LBH-123-AI/Comfyui_Minimax_h3_latent_Upscaler.git', 'Comfyui_Minimax_h3_latent_Upscaler'),
    'QwenTTS': ('https://github.com/flybirdxx/ComfyUI-Qwen-TTS.git', 'ComfyUI-Qwen-TTS'),
    'IndexTTS25': ('https://github.com/T8mars/comfyui-indextts25-t8.git', 'comfyui-indextts25-T8'),
    'BreezeTTS2': ('https://github.com/Saganaki22/ComfyUI-Breeze-TTS-2.git', 'ComfyUI-Breeze-TTS-2'),
    'MieNodes': ('https://github.com/MieMieeeee/ComfyUI-MieNodes.git', 'ComfyUI-MieNodes'),
}

FEATURES = {
    'h3_video': ('影片：MiniMax H3（一採／二採／Motion Context）', ['VHS', 'T8', 'MotionContext', 'KJNodes', 'ListHelper', 'GoohaiTools', 'LBH_Upscaler']),
    'motion': ('動作遷移／人物替換：SCAIL-2（官方節點 + VHS）', ['VHS']),
    'motion_mie': ('動作遷移 Mie loop 範例（只在 ComfyUI 手動開啟範例時需要）', ['VHS', 'KJNodes', 'MieNodes']),
    'image': ('圖片：Z-Image / Flux2 Klein / MOSS / Anima（官方節點）', []),
    'yue2': ('音樂：YuE2（官方節點）', []),
    'qwen_tts': ('TTS：Qwen3 TTS（模型由節點首次執行自動下載）', ['QwenTTS']),
    'index_tts': ('TTS：IndexTTS 2.5', ['IndexTTS25']),
    'breeze_tts': ('TTS：Breeze TTS 2（模型由節點首次執行自動下載）', ['BreezeTTS2']),
}

HF = 'hf'
# key: (feature, label, ComfyUI category folder, filename, source, approx GB, extra)
# source = ('hf', repo, path) | ('url', url) | ('snapshot', repo)
MODELS = {
    # MiniMax H3 ------------------------------------------------------------
    'h3_fl2va_int8': ('h3_video', 'H3 FL2VA int8（畫布預設 UNet）', 'diffusion_models', 'minimax_h3_fl2va_int8_convrot.safetensors', (HF, 'Comfy-Org/MiniMax-H3', 'diffusion_models/minimax_h3_fl2va_int8_convrot.safetensors'), 34.0, {}),
    'h3_fl2va_pruned_int8': ('h3_video', 'H3 FL2VA pruned int8', 'diffusion_models', 'minimax_h3_fl2va_pruned_int8_convrot.safetensors', (HF, 'Comfy-Org/MiniMax-H3', 'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors'), 21.0, {}),
    'h3_ref2va_pruned_int8': ('h3_video', 'H3 Ref2VA pruned int8', 'diffusion_models', 'minimax_h3_ref2va_pruned_int8_convrot.safetensors', (HF, 'Comfy-Org/MiniMax-H3', 'diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors'), 21.0, {}),
    'h3_hybrid_b25': ('h3_video', 'H3 Hybrid FL2VA+Ref2VA b25-49 int8（社群）', 'diffusion_models', 'minimax_h3_hybrid_fl2va_ref2va_b25-49-int8.safetensors', (HF, 'smhfacct/Minimax-H3-fl2va-ref2va-hybrid-models', 'minimax_h3_hybrid_fl2va_ref2va_b25-49-int8.safetensors'), 21.0, {}),
    'h3_text_encoder': ('h3_video', 'H3 Text Encoder Qwen3-VL 32B nvfp4', 'text_encoders', 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', (HF, 'Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'), 15.7, {}),
    'h3_video_vae': ('h3_video', 'H3 Video VAE int8', 'vae', 'minimax_h3_video_vae_int8_convrot.safetensors', (HF, 'Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_int8_convrot.safetensors'), 2.8, {}),
    'h3_audio_vae': ('h3_video', 'H3 Audio VAE fp32', 'vae', 'minimax_h3_audio_vae_fp32.safetensors', (HF, 'Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors'), 0.6, {}),
    'h3_lora_4step': ('h3_video', 'H3 Turbo 4-step LoRA v1.2 768p（lightx2v）', 'loras', 'minimax_h3_fl2v_turbo_4step_v1.2_768p_comfyui_bf16.safetensors', (HF, 'lightx2v/Minimax-h3-Turbo', 'minimax_h3_fl2v_turbo_4step_v1.2_768p_comfyui_bf16.safetensors'), 2.0, {}),
    'h3_lora_t8_4step': ('h3_video', 'H3 Turbo 4 步加速 LoRA（T8，畫布預設 LoRA）', 'loras', 'minimax_h3_turbo_4步加速_comfyui.safetensors', (HF, 't8star/minimax-h3-4step-turbo-loras-comfyui-exp', 'minimax_h3_turbo_4步加速_comfyui.safetensors'), 0.8, {}),
    'h3_lora_8step': ('h3_video', 'H3 Turbo 8-step LoRA v1.0（官方 8 步加速，選用）', 'loras', 'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors', (HF, 'Comfy-Org/MiniMax-H3', 'loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'), 2.0, {}),
    'pika_flex_fl2va': ('h3_video', 'Pika H3 Flex FL2VA LoRA（Pika 官方動態風格，選用）', 'loras', 'pika_h3_flex_fl2va_comfy_bf16.safetensors', (HF, 'Pika-Labs/Pika-H3-Flex', 'comfy/loras/pika_h3_flex_fl2va_comfy_bf16.safetensors'), 2.0, {}),
    'pika_flex_ref2va': ('h3_video', 'Pika H3 Flex Ref2VA LoRA（Pika 官方動態風格，搭配 Ref2VA 模型，選用）', 'loras', 'pika_h3_flex_ref2va_comfy_bf16.safetensors', (HF, 'Pika-Labs/Pika-H3-Flex', 'comfy/loras/pika_h3_flex_ref2va_comfy_bf16.safetensors'), 2.0, {}),
    'h3_upscaler': ('h3_video', 'H3 Latent Upscaler 3D conv v1（二採）', 'latent_upscale_models', 'minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors', (HF, 'LBH-123-AI/Minimax_h3_latent_Upscaler', 'minimax_h3_latent_upscaler_3d_conv_v1/minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors'), 0.3,
                    # The canvas two-pass default still names the older file; the
                    # previous notebook rewrote it to this file, so link that name.
                    {'aliases': ['minimax_h3_latent_upscaler_3d_fp16.safetensors']}),
    # SCAIL-2 motion transfer ----------------------------------------------
    'scail2_int8': ('motion', 'SCAIL-2 14B int8（畫布預設）', 'diffusion_models', 'wan2.1_14B_SCAIL_2_int8_convrot.safetensors', (HF, 'Comfy-Org/SCAIL-2', 'diffusion_models/wan2.1_14B_SCAIL_2_int8_convrot.safetensors'), 16.7, {}),
    'scail2_fp8': ('motion', 'SCAIL-2 14B fp8 scaled（選用）', 'diffusion_models', 'wan2.1_14B_SCAIL_2_fp8_scaled.safetensors', (HF, 'Comfy-Org/SCAIL-2', 'diffusion_models/wan2.1_14B_SCAIL_2_fp8_scaled.safetensors'), 17.7, {}),
    'scail2_dpo': ('motion', 'SCAIL-2 DPO LoRA', 'loras', 'wan2.1_SCAIL_2_DPO_lora_bf16.safetensors', (HF, 'Comfy-Org/SCAIL-2', 'loras/wan2.1_SCAIL_2_DPO_lora_bf16.safetensors'), 1.2, {}),
    'wan_distill_lora': ('motion', 'Wan2.1 lightx2v 加速 LoRA（官方 blueprint 來源）', 'loras', 'lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors', (HF, 'Kijai/WanVideo_comfy', 'Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors'), 0.7, {}),
    'wan_umt5': ('motion', 'Wan UMT5-XXL fp8 Text Encoder', 'text_encoders', 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', (HF, 'Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors'), 6.7, {}),
    'wan_clip_vision': ('motion', 'CLIP Vision H', 'clip_vision', 'clip_vision_h.safetensors', (HF, 'Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/clip_vision/clip_vision_h.safetensors'), 1.3, {}),
    'wan_vae': ('motion', 'Wan 2.1 VAE bf16', 'vae', 'Wan2_1_VAE_bf16.safetensors', (HF, 'Comfy-Org/Wan-Animate-2', 'vae/Wan2_1_VAE_bf16.safetensors'), 0.3, {}),
    'sam31': ('motion', 'SAM 3.1 multiplex（人物遮罩）', 'checkpoints', 'sam3.1_multiplex_fp16.safetensors', (HF, 'Comfy-Org/sam3.1', 'checkpoints/sam3.1_multiplex_fp16.safetensors'), 1.7, {}),
    # Images -----------------------------------------------------------------
    'z_image_turbo': ('image', 'Z-Image Turbo bf16', 'diffusion_models', 'z_image_turbo_bf16.safetensors', (HF, 'Comfy-Org/z_image_turbo', 'split_files/diffusion_models/z_image_turbo_bf16.safetensors'), 12.3, {}),
    'qwen3_4b': ('image', 'Qwen3 4B Text Encoder（Z-Image / Flux2 Klein）', 'text_encoders', 'qwen_3_4b.safetensors', (HF, 'Comfy-Org/z_image_turbo', 'split_files/text_encoders/qwen_3_4b.safetensors'), 8.0, {}),
    'z_image_vae': ('image', 'Z-Image VAE (ae)', 'vae', 'ae.safetensors', (HF, 'Comfy-Org/z_image_turbo', 'split_files/vae/ae.safetensors'), 0.3, {}),
    'flux2_klein_4b': ('image', 'Flux2 Klein 4B fp8（圖片編輯）', 'diffusion_models', 'flux-2-klein-4b-fp8.safetensors', (HF, 'black-forest-labs/FLUX.2-klein-4b-fp8', 'flux-2-klein-4b-fp8.safetensors'), 4.0, {}),
    'flux2_vae': ('image', 'Flux2 VAE（Flux2 Klein / MOSS）', 'vae', 'flux2-vae.safetensors', (HF, 'Comfy-Org/flux2-dev', 'split_files/vae/flux2-vae.safetensors'), 0.3, {}),
    'moss_sft_int8': ('image', 'MOSS Image SFT int8', 'diffusion_models', 'moss_image_sft_int8_convrot.safetensors', (HF, 'Comfy-Org/MOSS-Image', 'diffusion_models/moss_image_sft_int8_convrot.safetensors'), 5.4, {}),
    'moss_text_encoder': ('image', 'MOSS Qwen3.5 9B w4a8 Text Encoder', 'text_encoders', 'qwen3.5_9b_w4a8_gptq.safetensors', (HF, 'Comfy-Org/MOSS-Image', 'text_encoders/qwen3.5_9b_w4a8_gptq.safetensors'), 7.3, {}),
    'anima_base': ('image', 'Anima Base v1.0（畫布預設為社群微調，可另填名稱）', 'diffusion_models', 'anima-base-v1.0.safetensors', (HF, 'circlestone-labs/Anima', 'split_files/diffusion_models/anima-base-v1.0.safetensors'), 4.2, {}),
    'anima_text_encoder': ('image', 'Anima Qwen3 0.6B Text Encoder', 'text_encoders', 'qwen_3_06b_base.safetensors', (HF, 'circlestone-labs/Anima', 'split_files/text_encoders/qwen_3_06b_base.safetensors'), 1.2, {}),
    'anima_vae': ('image', 'Anima Qwen Image VAE', 'vae', 'qwen_image_vae.safetensors', (HF, 'circlestone-labs/Anima', 'split_files/vae/qwen_image_vae.safetensors'), 0.3, {}),
    # Audio -------------------------------------------------------------------
    'yue2_int8': ('yue2', 'YuE2 3B int8', 'checkpoints', 'yue2_3b_int8_convrot.safetensors', (HF, 'Comfy-Org/YuE2', 'checkpoints/yue2_3b_int8_convrot.safetensors'), 4.0, {}),
    'sheetsage2': ('yue2', 'SheetSage2 Audio Encoder（翻唱）', 'audio_encoders', 'sheetsage2_bf16.safetensors', (HF, 'Comfy-Org/YuE2', 'audio_encoders/sheetsage2_bf16.safetensors'), 1.4, {}),
    'indextts25': ('index_tts', 'IndexTTS 2.5 完整模型目錄', 'TTS', 'IndexTTS-2.5', ('snapshot', 't8star/IndexTTS-2.5-Comfy'), 5.0, {}),
}

# Optional extra downloads in the same shape, e.g. personal LoRAs:
# EXTRA_MODELS = [{'category':'loras','name':'my.safetensors','url':'https://...'}]
#                 [{'category':'diffusion_models','name':'x.safetensors','repo':'owner/repo','file':'path/x.safetensors'}]
EXTRA_MODELS = []
# Extra node packs: {'Name': 'https://github.com/owner/repo.git'}
CUSTOM_NODE_REPOS = {}


def _load_selection():
    try:
        return json.loads(SELECTION_PATH.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return {}


_saved = _load_selection()
_STYLE = {'description_width': 'initial'}
feature_boxes = {k: W.Checkbox(value=k in _saved.get('features', []), description=label, style=_STYLE, layout=W.Layout(width='98%')) for k, (label, _) in FEATURES.items()}
model_boxes, folder_boxes = {}, {}
for key, (feature, label, category, name, source, size, extra) in MODELS.items():
    model_boxes[key] = W.Checkbox(value=key in _saved.get('models', []), description=f'{label} · {size:g} GB', style=_STYLE, layout=W.Layout(width='60%'))
    folder_boxes[key] = W.Text(value=_saved.get('folders', {}).get(key, ''), placeholder=f'{category}（預設，留空）', layout=W.Layout(width='38%'))
root_box = W.Text(value=_saved.get('model_root', MODEL_ROOT), description='模型根目錄', style=_STYLE, layout=W.Layout(width='98%'))
summary = W.HTML()


def _refresh(*_):
    chosen = [k for k, b in model_boxes.items() if b.value]
    total = sum(MODELS[k][5] for k in chosen)
    packs = sorted({p for f, b in feature_boxes.items() if b.value for p in FEATURES[f][1]})
    summary.value = f'<b>已選 {len(chosen)} 個模型，約 {total:.1f} GB</b>；節點套件：{", ".join(packs) or "（僅 ComfyUI 核心）"}'


def _select_feature(feature):
    def handler(change):
        if change['new']:
            for key, spec in MODELS.items():
                # Pre-tick the canvas defaults of a feature; optional variants stay manual.
                if spec[0] == feature and not any(word in spec[1] for word in ('選用', '社群', 'pruned', 'Ref2VA', '8-step')):
                    model_boxes[key].value = True
        _refresh()
    return handler


def apply_selection(*_):
    global SELECTION
    SELECTION = {
        'features': [k for k, b in feature_boxes.items() if b.value],
        'models': [k for k, b in model_boxes.items() if b.value],
        'folders': {k: b.value.strip() for k, b in folder_boxes.items() if b.value.strip()},
        'model_root': root_box.value.strip() or str(COMFY / 'models'),
        'extra_models': EXTRA_MODELS,
        'custom_node_repos': CUSTOM_NODE_REPOS,
        'install_manager': INSTALL_MANAGER,
    }
    SELECTION_PATH.write_text(json.dumps(SELECTION, ensure_ascii=False, indent=2), encoding='utf-8')
    _refresh()
    print('✅ 已儲存選擇；接著執行 Cell 2（安裝）與 Cell 3（下載）。之後可隨時回來追加，再重跑 Cell 3。')


for key, box in feature_boxes.items():
    box.observe(_select_feature(key), names='value')
for box in model_boxes.values():
    box.observe(_refresh, names='value')
button = W.Button(description='✔ 儲存選擇', button_style='success')
button.on_click(apply_selection)
groups = []
for feature, (label, _) in FEATURES.items():
    rows = [W.HBox([model_boxes[k], folder_boxes[k]]) for k, v in MODELS.items() if v[0] == feature]
    groups.append(W.VBox([feature_boxes[feature], *rows], layout=W.Layout(border='1px solid #ddd', padding='4px', margin='4px 0')))
display(W.HTML('<b>勾選功能會安裝對應節點並預勾畫布預設模型</b>；可再個別取消。右欄可改單一模型的目標資料夾（相對於根目錄的類別資料夾，或絕對路徑）。'
               '<br>⚠️ 若改成類別底下的子資料夾（例如 diffusion_models/SCAIL-2），ComfyUI 的模型名稱會帶子資料夾前綴，畫布中需填入相同名稱。'),
        root_box, *groups, summary, button)
_refresh()
SELECTION = _saved or None

## Cell 2 — 安裝 ComfyUI 與節點
只安裝已勾選功能需要的套件。SCAIL-2、SAM3、YuE2、Z-Image、Flux2、MOSS、Anima 都是 ComfyUI 官方內建節點；動作遷移只額外需要 VideoHelperSuite。Cell 1 勾選 `INSTALL_MANAGER` 時會一併安裝 ComfyUI-Manager。

In [ ]:
#@title Cell 2 — 安裝 ComfyUI 與所選功能的節點套件
"""Installs ComfyUI core plus only the node packs required by the ticked features."""
import json
import re
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlparse

import torch

assert torch.cuda.is_available(), '請先在 Colab 選擇 GPU 執行階段（L4 / A100 / H100）。'
COMFY = Path('/content/ComfyUI')
SELECTION_PATH = Path('/content/h3_canvas_selection.json')
assert SELECTION_PATH.is_file(), '請先執行 Cell 1 並按「儲存選擇」。'
SELECTION = json.loads(SELECTION_PATH.read_text(encoding='utf-8'))
assert 'NODE_PACKS' in globals(), '請先執行 Cell 1。'

# Optional: pin versions with commit SHAs, e.g. {'ComfyUI': 'abc123', 'VHS': 'def456'}
REVISIONS = {}

if globals().get('COMFY_PROCESS') and COMFY_PROCESS.poll() is None:
    raise RuntimeError('ComfyUI 正在執行；新增節點前請重啟執行階段，或先停止 ComfyUI。')


def run(*args):
    subprocess.run(list(args), check=True)


def checkout(name, url, dest):
    if not (dest / '.git').is_dir():
        dest.parent.mkdir(parents=True, exist_ok=True)
        run('git', 'clone', '--depth', '1', url, str(dest))
    if name in REVISIONS:
        run('git', '-C', str(dest), 'fetch', '--depth', '1', 'origin', REVISIONS[name])
        run('git', '-C', str(dest), 'checkout', '--detach', 'FETCH_HEAD')
    requirements = dest / 'requirements.txt'
    if requirements.is_file():
        run(sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements))
    return subprocess.check_output(['git', '-C', str(dest), 'rev-parse', 'HEAD'], text=True).strip()


PROVENANCE = {'ComfyUI': checkout('ComfyUI', 'https://github.com/Comfy-Org/ComfyUI.git', COMFY)}
packs = sorted({p for f in SELECTION['features'] for p in FEATURES[f][1]})
repos = {name: NODE_PACKS[name] for name in packs}
for name, url in SELECTION.get('custom_node_repos', {}).items():
    parsed = urlparse(url)
    if not re.fullmatch(r'[A-Za-z0-9_-]+', name) or name in NODE_PACKS or parsed.scheme != 'https' or not parsed.netloc:
        raise ValueError('自訂節點名稱重複或網址不是 HTTPS Git 倉庫：' + name)
    repos[name] = (url, name)
for name, (url, folder) in repos.items():
    print('安裝節點：', name)
    PROVENANCE[name] = checkout(name, url, COMFY / 'custom_nodes' / folder)

# ComfyUI-Manager: newer ComfyUI ships it as a pip package enabled by --enable-manager;
# older checkouts use the classic custom_nodes clone. Cell 4 reads the chosen mode.
MANAGER_MODE = Path('/content/h3_manager_mode')
MANAGER_MODE.unlink(missing_ok=True)
if SELECTION.get('install_manager', True):
    print('安裝 ComfyUI-Manager')
    if (COMFY / 'manager_requirements.txt').is_file():
        run(sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'manager_requirements.txt'))
        MANAGER_MODE.write_text('pip', encoding='utf-8')
        PROVENANCE['ComfyUI-Manager'] = 'pip (manager_requirements.txt)'
    else:
        PROVENANCE['ComfyUI-Manager'] = checkout('ComfyUI-Manager', 'https://github.com/Comfy-Org/ComfyUI-Manager.git', COMFY / 'custom_nodes' / 'ComfyUI-Manager')
        MANAGER_MODE.write_text('legacy', encoding='utf-8')

run(sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'hf_xet', 'ipywidgets', 'requests',
    'fastapi', 'uvicorn', 'python-multipart', 'imageio-ffmpeg')
Path('/content/h3_canvas_provenance.json').write_text(json.dumps(PROVENANCE, indent=2), encoding='utf-8')
print('✅ 安裝完成：', ', '.join(PROVENANCE))
print('接著執行 Cell 3 下載所選模型。')

## Cell 3 — 下載模型
可重跑；已下載且來源相同會略過，下載失敗的項目不影響其他項目。H3 二採的 latent upscaler 會另外建立畫布預設檔名的相容連結。

In [ ]:
#@title Cell 3 — 下載所選模型（可重跑；已存在且來源相同會略過）
"""Download each ticked model into its ComfyUI category folder.

HF_TOKEN is read from Colab Secrets (or the environment) only when present; it is
never printed or written to disk. Gated repos need you to accept their licence on
Hugging Face first.
"""
import json
import os
import shutil
import subprocess
from pathlib import Path

from huggingface_hub import hf_hub_download, snapshot_download

COMFY = Path('/content/ComfyUI')
SELECTION = json.loads(Path('/content/h3_canvas_selection.json').read_text(encoding='utf-8'))
assert 'MODELS' in globals(), '請先執行 Cell 1。'
ROOT = Path(SELECTION['model_root']).expanduser()
STAGING = Path('/content/h3_model_staging')

HF_TOKEN = os.environ.get('HF_TOKEN') or None
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or None
    except Exception:
        HF_TOKEN = None


def target_dir(key, category):
    folder = SELECTION.get('folders', {}).get(key, '').strip()
    if not folder:
        return ROOT / category
    path = Path(folder).expanduser()
    return path if path.is_absolute() else ROOT / path


def safe_name(name):
    if not name or Path(name).name != name or name in ('.', '..'):
        raise ValueError('模型檔名不可包含路徑：' + repr(name))
    return name


def install(category, name, source, folder, aliases=()):
    name = safe_name(name)
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / name
    receipt = folder / (name + '.source.json')
    identity = json.dumps(source, sort_keys=True)
    if target.exists() and receipt.is_file() and receipt.read_text(encoding='utf-8') == identity:
        print('✓ 已存在：', target)
    elif source[0] == 'snapshot':
        snapshot_download(repo_id=source[1], local_dir=str(target), token=HF_TOKEN)
        receipt.write_text(identity, encoding='utf-8')
        print('✅ 目錄：', target)
    else:
        if target.exists():
            raise FileExistsError(f'{target} 已存在但來源不同；請改名或刪除後再下載。')
        partial = folder / (name + '.partial')
        if source[0] == 'hf':
            STAGING.mkdir(parents=True, exist_ok=True)
            cached = Path(hf_hub_download(repo_id=source[1], filename=source[2], token=HF_TOKEN, local_dir=str(STAGING)))
            shutil.move(str(cached), str(partial))
        elif source[0] == 'url':
            subprocess.run(['curl', '-fL', '--retry', '3', '-C', '-', '-o', str(partial), source[1]], check=True)
        else:
            raise ValueError('未知模型來源：' + source[0])
        partial.replace(target)
        receipt.write_text(identity, encoding='utf-8')
        print('✅ 下載：', target)
    for alias in aliases:
        link = folder / safe_name(alias)
        if not link.exists():
            link.symlink_to(target.name)
            print('   ↳ 相容名稱：', link.name)
    return folder


EXTRA_PATHS = {}
jobs = [(k, MODELS[k]) for k in SELECTION['models'] if k in MODELS]
for item in SELECTION.get('extra_models', []):
    source = ('url', item['url']) if item.get('url') else ('hf', item['repo'], item['file'])
    jobs.append((None, ('extra', item.get('name'), item['category'], item['name'], source, 0, {})))
failures = []
for key, (feature, label, category, name, source, size, extra) in jobs:
    print(f'— {label}（{size:g} GB）→ {category}')
    try:
        folder = install(category, name, source, target_dir(key, category) if key else ROOT / category, extra.get('aliases', ()))
        if not folder.resolve().is_relative_to((COMFY / 'models').resolve()):
            EXTRA_PATHS.setdefault(category, set()).add(str(folder))
    except Exception as exc:  # keep going so one gated repo does not stop the rest
        failures.append((label, f'{type(exc).__name__}: {exc}'))
        print('❌', label, failures[-1][1][:300])

# Folders outside ComfyUI/models become an extra_model_paths.yaml entry (Cell 4).
yaml_path = Path('/content/h3_extra_model_paths.yaml')
if EXTRA_PATHS:
    lines = ['h3_canvas:']
    for category, folders in sorted(EXTRA_PATHS.items()):
        lines.append(f'  {category}: |')
        lines += [f'    {folder}' for folder in sorted(folders)]
    yaml_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    print('已寫入額外模型路徑：', yaml_path)
elif yaml_path.exists():
    yaml_path.unlink()
shutil.rmtree(STAGING, ignore_errors=True)
print('\n完成；失敗', len(failures), '項。' if failures else '項。接著執行 Cell 4 啟動 ComfyUI。')
for label, reason in failures:
    print(' -', label, '：', reason[:300])

## Cell 4 — 啟動 ComfyUI
ComfyUI 只綁定 127.0.0.1；外部只能透過 Cell 5 的 token API。啟動參數含 `--enable-cors-header` 與 `--enable-compress-response-body`，讓最後一格的 ComfyUI 網頁介面能正常開啟；有安裝新版 Manager 時自動加上 `--enable-manager`。用 Manager 裝完新套件後，重跑本 Cell 即可載入。啟動後會列出各已選功能是否缺少節點。

In [ ]:
#@title Cell 4 — 啟動 ComfyUI（只綁 127.0.0.1，外部只能經過 Cell 5 的認證 API）
import json
import subprocess
import sys
import time
from pathlib import Path

import requests

COMFY = Path('/content/ComfyUI')
COMFY_URL = 'http://127.0.0.1:8188'
SELECTION = json.loads(Path('/content/h3_canvas_selection.json').read_text(encoding='utf-8'))
#@markdown 額外 ComfyUI 啟動參數（例如 --lowvram、--reserve-vram 2）
EXTRA_ARGS = ''  #@param {type:'string'}

if globals().get('API_SERVER') is not None and globals().get('API_THREAD') and API_THREAD.is_alive():
    print('提醒：Cell 5 的 API 仍在執行；ComfyUI 重啟期間的請求會暫時失敗。')
if globals().get('COMFY_PROCESS') and COMFY_PROCESS.poll() is None:
    COMFY_PROCESS.terminate()
    try:
        COMFY_PROCESS.wait(timeout=30)
    except subprocess.TimeoutExpired:
        COMFY_PROCESS.kill()

# CORS + compression let the ComfyUI web page (optional cell below) open through the Colab proxy.
command = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--disable-auto-launch',
           '--enable-cors-header', '--enable-compress-response-body']
manager_mode = Path('/content/h3_manager_mode')
if manager_mode.is_file() and manager_mode.read_text(encoding='utf-8').strip() == 'pip':
    command.append('--enable-manager')
yaml_path = Path('/content/h3_extra_model_paths.yaml')
if yaml_path.is_file():
    command += ['--extra-model-paths-config', str(yaml_path)]
command += EXTRA_ARGS.split()
COMFY_LOG = Path('/content/comfyui.log')
COMFY_PROCESS = subprocess.Popen(command, cwd=COMFY, stdout=COMFY_LOG.open('w'), stderr=subprocess.STDOUT)
session = requests.Session()
session.trust_env = False
for second in range(600):
    if COMFY_PROCESS.poll() is not None:
        print(COMFY_LOG.read_text(errors='replace')[-6000:])
        raise RuntimeError('ComfyUI 啟動失敗，請查看上方 log。')
    try:
        if session.get(COMFY_URL + '/system_stats', timeout=3).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise TimeoutError('ComfyUI 10 分鐘內未就緒。')

# Report missing node classes for each ticked feature (non-blocking).
REQUIRED_NODES = {
    'h3_video': ['MiniMaxH3AudioConditioningT8', 'MiniMaxH3MotionContext', 'MiniMaxLowVRAMAttention', 'VHS_VideoCombine', 'ResolutionSelector', 'MiniMaxH3LearnedLatentUpscaleT8Advanced'],
    'motion': ['WanSCAILToVideo', 'SCAIL2ColoredMask', 'SAM3_VideoTrack', 'ColorTransfer', 'BatchImagesNode', 'ResizeAndPadImage', 'VHS_LoadVideo', 'VHS_VideoInfoLoaded', 'VHS_VideoCombine'],
    'image': ['EmptyFlux2LatentImage', 'ReferenceLatent', 'SkipLayerGuidanceDiTSimple', 'ResolutionSelector'],
    'yue2': ['YuE2GenerateMusic', 'SheetSage2AudioToABC', 'SeedNode', 'SaveAudioAdvanced'],
    'qwen_tts': ['FB_Qwen3TTSVoiceClone', 'FB_Qwen3TTSVoiceDesign'],
    'index_tts': ['T8_IndexTTS25_Generate', 'T8_IndexTTS25_ModelLoader', 'PrimitiveFloat'],
    'breeze_tts': ['BreezeTTS2LoadModel'],
}
info = session.get(COMFY_URL + '/object_info', timeout=120).json()
for feature in SELECTION['features']:
    missing = [n for n in REQUIRED_NODES.get(feature, []) if n not in info]
    print(('⚠️ ' if missing else '✅ ') + FEATURES[feature][0] + ('：缺少 ' + ', '.join(missing) if missing else ''))
print('ComfyUI 已就緒（log：/content/comfyui.log）。接著執行 Cell 5 建立畫布 API。')

## Cell 5 — 畫布 API 橋接與 HTTPS tunnel
提供與 ComfyUI 原生相容的端點（全部需要 `Authorization: Bearer <H3_API_TOKEN>`）：
`/object_info`、`/prompt`、`/history/{id}`、`/queue`、`/interrupt`、`/upload/image`、`/canvas/upload/video`、`/upload/audio`、`/view`、`/canvas/cancel/{id}`，以及 `/health`、`/heartbeat`、`/models`。

在雪音畫布「連線設定 → 生成服務來源」新增或編輯 **Colab** 來源，填入 `H3_API_URL` 與 token，然後在任何節點的「運算服務」選擇該來源即可。tunnel 網址每次重建都會改變。

In [ ]:
#@title Cell 5 — 啟動畫布 API 橋接（Bearer token）與 HTTPS tunnel
"""Authenticated native-ComfyUI bridge for the H3 canvas.

The canvas compiles complete API graphs itself and submits them here, exactly
like a RunningHub/ComfyUI backend, so no workflow JSON has to be uploaded to
Colab. All routes require `Authorization: Bearer <H3_API_TOKEN>`.
"""
import asyncio
import getpass
import hmac
import os
import re
import secrets
import shutil
import subprocess
import tempfile
import threading
import time
import uuid
from pathlib import Path

import requests
import uvicorn
from fastapi import Body, Depends, FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse

COMFY = Path('/content/ComfyUI')
COMFY_URL = 'http://127.0.0.1:8188'
PROTOCOL = 'h3-canvas-bridge-1'
API_HOST, API_PORT = '127.0.0.1', 7860
LOCAL_API_URL = f'http://{API_HOST}:{API_PORT}'
TUNNEL_LOG = Path('/content/h3_api_tunnel.log')
UPLOAD_LIMIT = 1024 * 2**20
OUTPUT_SUFFIXES = {'.mp4', '.webm', '.mov', '.gif', '.png', '.jpg', '.jpeg', '.webp', '.wav', '.mp3', '.flac', '.ogg',
                   '.safetensors', '.latent', '.zip', '.glb', '.gltf', '.obj', '.ply', '.stl'}
#@markdown 允許瀏覽器直接跨網域呼叫的來源（逗號分隔；本機畫布由後端代理呼叫，可留空）
FRONTEND_ALLOWED_ORIGINS = ''  #@param {type:'string'}
#@markdown cloudflared 協定（http2 走 TCP，較穩；也可填 auto / quic）
TUNNEL_PROTOCOL = 'http2'  #@param ['http2', 'auto', 'quic']

assert globals().get('COMFY_PROCESS') and COMFY_PROCESS.poll() is None, '請先執行 Cell 4 啟動 ComfyUI。'

# Same token survives re-running this cell. Sources: runtime → env → Colab Secrets → prompt.
_token = globals().get('API_TOKEN') or os.environ.get('H3_API_TOKEN', '')
if not _token:
    try:
        from google.colab import userdata
        _token = userdata.get('H3_API_TOKEN') or ''
    except Exception:
        pass
_generated = False
if not _token:
    _token = getpass.getpass('H3_API_TOKEN（至少 24 字元；直接按 Enter 會自動產生）：').strip()
    if not _token:
        _token, _generated = secrets.token_urlsafe(32), True
if len(_token) < 24:
    raise ValueError('H3_API_TOKEN 必須至少 24 字元。')
API_TOKEN = _token
del _token


def _redact(value):
    return str(value).replace(API_TOKEN, '[TOKEN REDACTED]')


def require_token(authorization: str = Header(default='')):
    scheme, _, supplied = authorization.partition(' ')
    if scheme.lower() != 'bearer' or not hmac.compare_digest(supplied.strip().encode(), API_TOKEN.encode()):
        raise HTTPException(401, '需要有效的 Bearer token', headers={'WWW-Authenticate': 'Bearer'})


def comfy(method, path, **kwargs):
    try:
        response = requests.request(method, COMFY_URL + path, timeout=(15, 300), **kwargs)
    except requests.RequestException as exc:
        raise HTTPException(502, 'ComfyUI 無法連線：' + _redact(exc)) from None
    try:
        body = response.json() if response.content else {}
    except ValueError:
        body = {'detail': response.text[:4000]}
    # Pass ComfyUI validation errors (400 + node_errors) through unchanged.
    return JSONResponse(body, status_code=response.status_code)


def _check_id(prompt_id):
    if not re.fullmatch(r'[A-Za-z0-9_-]{1,100}', prompt_id):
        raise HTTPException(422, '無效的 prompt_id')


def create_bridge():
    app = FastAPI(title='H3 Canvas Colab Bridge', dependencies=[Depends(require_token)], docs_url=None, redoc_url=None)
    origins = [x.strip() for x in (FRONTEND_ALLOWED_ORIGINS or os.environ.get('H3_ALLOWED_ORIGINS', '')).split(',') if x.strip()]
    if origins:
        app.add_middleware(CORSMiddleware, allow_origins=origins, allow_methods=['*'], allow_headers=['Authorization', 'Content-Type'])

    @app.get('/_h3_ready')
    def ready():
        return {'protocol': PROTOCOL, 'ready': True}

    @app.get('/health')
    async def health():
        try:
            stats = await asyncio.to_thread(lambda: requests.get(COMFY_URL + '/system_stats', timeout=10).json())
            return {'protocol': PROTOCOL, 'ready': True, 'bridge': 'universal', 'comfyui': stats.get('system', {}), 'devices': stats.get('devices', [])}
        except Exception as exc:
            return JSONResponse({'protocol': PROTOCOL, 'ready': False, 'error': _redact(exc)}, status_code=503)

    @app.get('/heartbeat')
    async def heartbeat():
        queue = await asyncio.to_thread(lambda: requests.get(COMFY_URL + '/queue', timeout=10).json())
        return {'ok': True, 'protocol': PROTOCOL, 'running': len(queue.get('queue_running', [])), 'pending': len(queue.get('queue_pending', [])), 'time': time.time()}

    @app.get('/object_info')
    async def object_info():
        return await asyncio.to_thread(comfy, 'GET', '/object_info')

    @app.get('/object_info/{node_class}')
    async def object_info_node(node_class: str):
        return await asyncio.to_thread(comfy, 'GET', '/object_info/' + requests.utils.quote(node_class, safe=''))

    @app.post('/prompt')
    async def prompt(payload: dict = Body(...)):
        if not isinstance(payload.get('prompt'), dict) or not payload['prompt']:
            raise HTTPException(422, '缺少 prompt API JSON')
        return await asyncio.to_thread(comfy, 'POST', '/prompt', json={'prompt': payload['prompt'], 'client_id': str(payload.get('client_id') or uuid.uuid4().hex)})

    @app.get('/history/{prompt_id}')
    async def history(prompt_id: str):
        _check_id(prompt_id)
        return await asyncio.to_thread(comfy, 'GET', '/history/' + prompt_id)

    @app.get('/queue')
    async def queue():
        return await asyncio.to_thread(comfy, 'GET', '/queue')

    @app.post('/queue')
    async def queue_change(payload: dict = Body(...)):
        return await asyncio.to_thread(comfy, 'POST', '/queue', json=payload)

    @app.post('/interrupt')
    async def interrupt(payload: dict = Body(default={})):
        return await asyncio.to_thread(comfy, 'POST', '/interrupt', json=payload)

    @app.post('/canvas/cancel/{prompt_id}')
    async def cancel(prompt_id: str):
        _check_id(prompt_id)
        def work():
            queue = requests.get(COMFY_URL + '/queue', timeout=15).json()
            requests.post(COMFY_URL + '/queue', json={'delete': [prompt_id]}, timeout=15).raise_for_status()
            if any(str(item[1]) == prompt_id for item in queue.get('queue_running', [])):
                requests.post(COMFY_URL + '/interrupt', json={'prompt_id': prompt_id}, timeout=15).raise_for_status()
        try:
            await asyncio.to_thread(work)
        except Exception:
            raise HTTPException(502, '取消失敗') from None
        return {'ok': True}

    async def forward_upload(upload, kind, folder_type='input', subfolder=''):
        if folder_type not in ('input', 'output'):
            raise HTTPException(422, '只接受 input / output')
        if subfolder and (Path(subfolder).is_absolute() or '..' in Path(subfolder).parts):
            raise HTTPException(422, '無效的子資料夾')
        suffix = Path(upload.filename or '').suffix.lower()
        if not re.fullmatch(r'\.[a-z0-9]{1,12}', suffix or ''):
            raise HTTPException(422, '缺少有效副檔名')
        name = f'canvas_{kind}_{uuid.uuid4().hex}{suffix}'
        total = 0
        with tempfile.SpooledTemporaryFile(max_size=64 * 2**20) as spool:
            while block := await upload.read(1024 * 1024):
                total += len(block)
                if total > UPLOAD_LIMIT:
                    raise HTTPException(413, '上傳檔案超過 1 GB')
                spool.write(block)
            await upload.close()
            if not total:
                raise HTTPException(422, '不能上傳空檔案')
            spool.seek(0)
            # ComfyUI's own upload endpoint stores any media (image/video/audio/latent).
            response = await asyncio.to_thread(lambda: requests.post(
                COMFY_URL + '/upload/image', files={'image': (name, spool, 'application/octet-stream')},
                data={'type': folder_type, 'subfolder': subfolder, 'overwrite': 'false'}, timeout=(15, 600)))
        if not response.ok:
            raise HTTPException(response.status_code, response.text[:2000])
        return response.json()

    @app.post('/upload/image')
    async def upload_image(image: UploadFile = File(...), type: str = Form('input'), subfolder: str = Form(''), overwrite: str = Form('false')):
        return await forward_upload(image, 'image', type, subfolder)

    @app.post('/canvas/upload/video')
    async def upload_video(video: UploadFile = File(...)):
        return await forward_upload(video, 'video')

    @app.post('/upload/audio')
    async def upload_audio(audio: UploadFile = File(...)):
        return await forward_upload(audio, 'audio')

    def output_file(filename, subfolder='', folder_type='output'):
        if folder_type not in ('output', 'temp'):
            raise HTTPException(403, '僅提供生成輸出')
        base = (COMFY / folder_type).resolve()
        path = (base / subfolder / filename).resolve()
        if not path.is_relative_to(base) or not path.is_file() or path.suffix.lower() not in OUTPUT_SUFFIXES:
            raise HTTPException(404, '找不到輸出')
        return FileResponse(path, filename=path.name)

    @app.get('/view')
    def view(filename: str, subfolder: str = '', type: str = 'output'):
        return output_file(filename, subfolder, type)

    @app.get('/canvas/result/{prompt_id}/{pass_name}')
    async def legacy_result(prompt_id: str, pass_name: str):
        """Compatibility with canvases built for the 20261102 notebook (H3 nodes 356/357)."""
        _check_id(prompt_id)
        node = {'first': '356', 'upscaled': '357', 'second': '357'}.get(pass_name)
        if not node:
            raise HTTPException(404, '請使用 first 或 upscaled')
        entry = (await asyncio.to_thread(lambda: requests.get(COMFY_URL + '/history/' + prompt_id, timeout=30).json())).get(prompt_id) or {}
        for items in entry.get('outputs', {}).get(node, {}).values():
            for item in items if isinstance(items, list) else []:
                if isinstance(item, dict) and str(item.get('filename', '')).lower().endswith('.mp4'):
                    return output_file(item['filename'], item.get('subfolder', ''), item.get('type', 'output'))
        raise HTTPException(404, '尚無輸出影片')

    @app.get('/models')
    def models():
        categories = {}
        roots = [COMFY / 'models']
        extra = Path('/content/h3_extra_model_paths.yaml')
        for category_dir in (d for root in roots for d in root.iterdir() if d.is_dir()):
            names = sorted(str(p.relative_to(category_dir)) for p in category_dir.rglob('*') if p.is_file() and p.suffix in ('.safetensors', '.ckpt', '.pt', '.pth', '.gguf', '.bin'))
            if names:
                categories[category_dir.name] = names
        return {'categories': categories, 'extra_model_paths': extra.read_text() if extra.exists() else ''}

    return app


def _stop_tunnel(process):
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()


_stop_tunnel(globals().get('TUNNEL_PROCESS'))
if globals().get('API_SERVER') is not None:
    API_SERVER.should_exit = True
    if globals().get('API_THREAD') is not None:
        API_THREAD.join(timeout=15)
        if API_THREAD.is_alive():
            raise RuntimeError('前一次 API 尚未停止，請稍後重跑本格。')

APP = create_bridge()
API_SERVER = uvicorn.Server(uvicorn.Config(APP, host=API_HOST, port=API_PORT, log_level='warning', access_log=False))
API_THREAD = threading.Thread(target=API_SERVER.run, daemon=True)
API_THREAD.start()
for _ in range(150):
    if API_SERVER.started:
        break
    if not API_THREAD.is_alive():
        raise RuntimeError('本機 API 啟動失敗；請確認 7860 埠沒有被占用。')
    time.sleep(0.1)
else:
    raise TimeoutError('本機 API 啟動超過 15 秒。')

local = requests.Session()
local.trust_env = False
headers = {'Authorization': 'Bearer ' + API_TOKEN}
assert local.get(LOCAL_API_URL + '/_h3_ready', timeout=10).status_code == 401, '未帶 token 的請求應被拒絕'
health = local.get(LOCAL_API_URL + '/health', headers=headers, timeout=30)
if not health.ok:
    raise RuntimeError('本機 /health 失敗：' + _redact(health.text[:500]))
print('1/2 本機 API、token 與 ComfyUI 檢查通過。')

cloudflared = Path('/content/cloudflared')
if not cloudflared.is_file():
    subprocess.run(['curl', '-fL', '--retry', '3', '--connect-timeout', '15',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-o', str(cloudflared)], check=True)
cloudflared.chmod(0o755)
with TUNNEL_LOG.open('w', encoding='utf-8') as stream:
    TUNNEL_PROCESS = subprocess.Popen([str(cloudflared), 'tunnel', '--no-autoupdate', '--protocol', TUNNEL_PROTOCOL, '--url', LOCAL_API_URL],
                                      stdout=stream, stderr=subprocess.STDOUT)
H3_API_URL = ''
deadline = time.monotonic() + 90
while time.monotonic() < deadline and not H3_API_URL:
    if TUNNEL_PROCESS.poll() is not None:
        raise RuntimeError('cloudflared 已退出：\n' + _redact(TUNNEL_LOG.read_text(errors='replace')[-4000:]))
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', TUNNEL_LOG.read_text(encoding='utf-8', errors='replace'))
    H3_API_URL = match.group(0) if match else ''
    time.sleep(0.5)
if not H3_API_URL:
    _stop_tunnel(TUNNEL_PROCESS)
    raise TimeoutError('90 秒內未取得 tunnel 網址。')

print('2/2 驗證公開網址（剛建立時可能需要等待）……')
verified = False
public = requests.Session()
for attempt in range(60):
    try:
        response = public.get(H3_API_URL + '/_h3_ready', headers=headers, timeout=(5, 10), allow_redirects=False)
        if response.ok and response.json().get('protocol') == PROTOCOL:
            verified = True
            break
    except (requests.RequestException, ValueError):
        pass
    time.sleep(2)
print('✅ 公開 API 驗證成功' if verified else '⚠️ 尚未從 Colab 驗證公開網址；tunnel 保留，請從畫布按「儲存」後測試。')
print('\n在畫布「連線設定 → 生成服務來源 → Colab」填入：')
print('  H3_API_URL   =', H3_API_URL)
if _generated:
    print('  H3_API_TOKEN =', API_TOKEN)
    print('  ⚠️ 這是自動產生的 token；複製後請清除本格輸出，不要分享 notebook 輸出。')
else:
    print('  H3_API_TOKEN = 你提供的那一組（本格不顯示）')
print('畫布會直接送出組裝好的 API 圖；不需要在 Colab 上傳任何工作流 JSON。')

## Cell 6 — 狀態與（選用）ComfyUI 介面
最後一格會開啟 ComfyUI 網頁介面：可手動載入 example_workflows 檢查，或用 Manager 安裝其他節點套件（裝完重跑 Cell 4）。

In [ ]:
#@title Cell 6 — 狀態檢查（心跳、佇列、模型清單；可隨時重跑）
import requests
assert globals().get('API_SERVER') and API_SERVER.started, '請先執行 Cell 5'
with requests.Session() as s:
    s.trust_env = False
    h = {'Authorization': 'Bearer ' + API_TOKEN}
    print('心跳：', s.get(LOCAL_API_URL + '/heartbeat', headers=h, timeout=10).json())
    models = s.get(LOCAL_API_URL + '/models', headers=h, timeout=60).json()['categories']
    for category, names in sorted(models.items()):
        print(f'[{category}]', ', '.join(names[:12]) + (' …' if len(names) > 12 else ''))
print('\nH3_API_URL =', H3_API_URL)
print('提醒：心跳只監控連線，不能防止 Colab 閒置回收。')

In [ ]:
#@title （選用）開啟 ComfyUI 網頁介面（檢查範例工作流、用 Manager 裝套件）
from google.colab import output
print('ComfyUI 介面僅供本人使用；雪音畫布不需要它。可拖入 repo 的 example_workflows/*.json，或從 Manager 安裝其他節點（裝完重跑 Cell 4）。')
output.serve_kernel_port_as_window(8188)